In [1]:
%pip install requests pandas


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests
import pandas as pd

# TRAFFIC DATA
traffic_url = "https://data.cityofnewyork.us/resource/i4gi-tjb9.json"

traffic_response = requests.get(
    traffic_url,
    params={"$limit": 5000},
    timeout=30
)
traffic_response.raise_for_status()

traffic = pd.DataFrame(traffic_response.json())

print("TRAFFIC")
print("Rows retrieved:", len(traffic))
print("Columns:", traffic.columns.tolist())
display(traffic.head())

# Check how much history is actually available.
coverage_response = requests.get(
    traffic_url,
    params={
        "$select": (
            "min(data_as_of) as earliest,"
            "max(data_as_of) as latest,"
            "count(*) as total_records"
        )
    },
    timeout=30
)
coverage_response.raise_for_status()

print("TRAFFIC DATA COVERAGE")
print(coverage_response.json())

# WEATHER DATA 
weather_url = "https://api.open-meteo.com/v1/forecast"

weather_response = requests.get(
    weather_url,
    params={
        "latitude": 40.7128,
        "longitude": -74.0060,
        "hourly": "temperature_2m,rain,precipitation",
        "past_days": 2,
        "forecast_days": 1,
        "timezone": "America/New_York"
    },
    timeout=30
)
weather_response.raise_for_status()

weather_json = weather_response.json()
weather = pd.DataFrame(weather_json["hourly"])

print("\nWEATHER")
print("Rows retrieved:", len(weather))
print("Units:", weather_json["hourly_units"])
display(weather.head())

TRAFFIC
Rows retrieved: 5000
Columns: ['id', 'speed', 'travel_time', 'status', 'data_as_of', 'link_id', 'link_points', 'encoded_poly_line', 'encoded_poly_line_lvls', 'owner', 'transcom_id', 'borough', 'link_name']


,id,speed,travel_time,status,data_as_of,link_id,link_points,encoded_poly_line,encoded_poly_line_lvls,owner,transcom_id,borough,link_name
0,211,37.9,435,0,2026-11-01T17:13:06.000,4362250,"40.78795,-73.790191 40.78647,-73.78812 40.7862...",uk}wFtckaMfH}Kr@cAbAoAzAyAvCeCt@e@\\|EwBxBg@jR...,BBBBBBBBBBBBBBBBBBBBBBBBBB,NYC-DOT-Region 10,4362250,Queens,CVE NB GCP - WILLETS PT BLVD
1,405,18.64,143,0,2026-11-01T17:13:06.000,4362314,"40.79782,-73.79339 40.793425,-73.79344 40.7927...",ki_xFtwkaMnZH\\|BM\\|Ce@bFsA??vCo@bJm@t@a@X]Xs...,BBBBBBBBBBBBBBBB,NYC_DOT_LIC,4362314,Queens,TNB S Qns Anchorage - CIP S @ TNB
2,311,18.64,266,0,2026-11-01T17:13:06.000,4362342,"40.72966,-73.87775 40.730165,-73.87643 40.7304...",k_rwF\\|f\\|aMcBgGo@wCqK{g@uJ_d@wCuOuBuJ}D_TgA...,BBBBBBBBBB,NYC_DOT_LIC,4362342,Queens,LIE E 84TH ST - 108TH ST
3,212,54.05,201,0,2026-11-01T17:13:03.000,4362244,"40.78802,-73.79003 40.7868604,-73.78838 40.786...",cl}wFtbkaMfFiIdByBrD{DtAeAj@_@nA}@\\|BcAvBy@hB...,BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBB...,NYC-DOT-Region 10,4362244,Queens,CVE NB LIE - WILLETS PT BLVD
4,318,22.36,175,0,2026-11-01T17:13:03.000,4362249,"40.7442206,-73.771661 40.7454306,-73.76907 40....",kztwFzogaMqFeOu@wC_DeQcFwXyAoF{CkIaAmDu@aEiDeT...,BBBBBBBBBBBBBBBBBBBBBBBBBBBBBB,NYC-DOT-Region 10,4362249,Queens,LIE WB LITTLE NECK PKWY - NB CIP


ReadTimeout: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)

In [4]:
# Inspect the downloaded sample 
traffic_dates = pd.to_datetime(traffic["data_as_of"], errors="coerce")

print("DOWNLOADED TRAFFIC SAMPLE")
print("Rows:", len(traffic))
print("Earliest timestamp:", traffic_dates.min())
print("Latest timestamp:", traffic_dates.max())
print("Unique timestamps:", traffic_dates.nunique())
print("Unique road segments:", traffic["link_id"].nunique())

display(
    traffic[
        ["link_id", "speed", "travel_time", "data_as_of"]
    ].head()
)

DOWNLOADED TRAFFIC SAMPLE
Rows: 5000
Earliest timestamp: 2026-11-01 13:53:00
Latest timestamp: 2026-11-01 17:13:06
Unique timestamps: 336
Unique road segments: 125


,link_id,speed,travel_time,data_as_of
0,4362250,37.9,435,2026-11-01T17:13:06.000
1,4362314,18.64,143,2026-11-01T17:13:06.000
2,4362342,18.64,266,2026-11-01T17:13:06.000
3,4362244,54.05,201,2026-11-01T17:13:03.000
4,4362249,22.36,175,2026-11-01T17:13:03.000


In [5]:
weather_response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": 40.7128,
        "longitude": -74.0060,
        "hourly": "temperature_2m,rain,precipitation",
        "past_days": 2,
        "forecast_days": 1,
        "timezone": "America/New_York"
    },
    timeout=30
)
weather_response.raise_for_status()

weather_json = weather_response.json()
weather = pd.DataFrame(weather_json["hourly"])

print("WEATHER")
print("Rows:", len(weather))
print("Units:", weather_json["hourly_units"])
display(weather.head())

WEATHER
Rows: 72
Units: {'time': 'iso8601', 'temperature_2m': '°C', 'rain': 'mm', 'precipitation': 'mm'}


,time,temperature_2m,rain,precipitation
0,2026-10-30T00:00,14.2,0.0,0.0
1,2026-10-30T01:00,14.0,0.0,0.0
2,2026-10-30T02:00,13.9,0.0,0.0
3,2026-10-30T03:00,13.8,0.0,0.0
4,2026-10-30T04:00,13.5,0.0,0.0


In [6]:
# Check whether the API contains traffic records before November 1.
try:
    history_response = requests.get(
        "https://data.cityofnewyork.us/resource/i4gi-tjb9.json",
        params={
            "$select": "link_id,speed,travel_time,data_as_of",
            "$where": "data_as_of < '2026-11-01T00:00:00'",
            "$limit": 1
        },
        timeout=30
    )
    history_response.raise_for_status()

    older_records = history_response.json()

    if older_records:
        print("Older traffic data found:")
        display(pd.DataFrame(older_records))
    else:
        print("No traffic records before November 1 were returned.")

except requests.exceptions.RequestException as error:
    print("History check failed:", error)
    print("This does not mean historical data is unavailable.")

Older traffic data found:


,link_id,speed,travel_time,data_as_of
0,4616253,44.73,56,2026-10-31T23:58:12.000


In [7]:
# Check how far back the traffic data extends.
check_dates = ["2026-10-25", "2026-10-01", "2026-09-01"]

for check_date in check_dates:
    try:
        response = requests.get(
            "https://data.cityofnewyork.us/resource/i4gi-tjb9.json",
            params={
                "$select": "link_id,speed,travel_time,data_as_of",
                "$where": f"data_as_of < '{check_date}T00:00:00'",
                "$limit": 1
            },
            timeout=15
        )
        response.raise_for_status()
        records = response.json()

        if records:
            print(
                f"Before {check_date}: FOUND — "
                f"{records[0]['data_as_of']}"
            )
        else:
            print(f"Before {check_date}: No records returned")

    except requests.exceptions.RequestException:
        print(f"Before {check_date}: Request failed; coverage unknown")

Before 2026-10-25: FOUND — 2026-10-24T23:58:12.000
Before 2026-10-01: FOUND — 2026-09-30T23:58:12.000
Before 2026-09-01: FOUND — 2026-08-31T23:58:12.000


In [8]:
# Retrieve one week for the road segment found in our history check.
# Request one day at a time
traffic_url = "https://data.cityofnewyork.us/resource/i4gi-tjb9.json"

segment_id = "4616253"
daily_frames = []

for day in pd.date_range("2026-10-19", "2026-10-25"):
    next_day = day + pd.Timedelta(days=1)
    offset = 0

    while True:
        response = requests.get(
            traffic_url,
            params={
                "$select": (
                    "link_id,link_name,borough,speed,"
                    "travel_time,status,data_as_of"
                ),
                "$where": (
                    f"link_id = '{segment_id}' "
                    f"AND data_as_of >= '{day:%Y-%m-%d}T00:00:00' "
                    f"AND data_as_of < '{next_day:%Y-%m-%d}T00:00:00'"
                ),
                "$order": "data_as_of, :id",
                "$limit": 1000,
                "$offset": offset
            },
            timeout=30
        )
        response.raise_for_status()
        records = response.json()

        if not records:
            break

        daily_frames.append(pd.DataFrame(records))
        offset += len(records)

        if len(records) < 1000:
            break

    print(f"{day:%Y-%m-%d}: {offset} records")

if daily_frames:
    traffic_week = pd.concat(daily_frames, ignore_index=True)

    traffic_week["data_as_of"] = pd.to_datetime(
        traffic_week["data_as_of"], errors="coerce"
    )
    traffic_week["speed"] = pd.to_numeric(
        traffic_week["speed"], errors="coerce"
    )

    print("\nTOTAL RECORDS:", len(traffic_week))
    print("ROAD:", traffic_week["link_name"].iloc[0])
    print("BOROUGH:", traffic_week["borough"].iloc[0])
    print("FIRST:", traffic_week["data_as_of"].min())
    print("LAST:", traffic_week["data_as_of"].max())
    print("MISSING SPEEDS:", traffic_week["speed"].isna().sum())
    print("STATUS COUNTS:")
    print(traffic_week["status"].value_counts(dropna=False))

    display(traffic_week.head())
else:
    print("No records found for this segment during the selected week.")

2026-10-19: 288 records
2026-10-20: 288 records
2026-10-21: 288 records
2026-10-22: 288 records
2026-10-23: 288 records
2026-10-24: 288 records
2026-10-25: 287 records

TOTAL RECORDS: 2015
ROAD: CBE E TAYLOR AVENUE - CASTLE HILL AVENUE
BOROUGH: Bronx
FIRST: 2026-10-19 00:03:09
LAST: 2026-10-25 23:58:10
MISSING SPEEDS: 0
STATUS COUNTS:
status
0       2011
-101       4
Name: count, dtype: int64


,link_id,link_name,borough,speed,travel_time,status,data_as_of
0,4616253,CBE E TAYLOR AVENUE - CASTLE HILL AVENUE,Bronx,50.33,50,0,2026-10-19 00:03:09
1,4616253,CBE E TAYLOR AVENUE - CASTLE HILL AVENUE,Bronx,44.73,56,0,2026-10-19 00:08:09
2,4616253,CBE E TAYLOR AVENUE - CASTLE HILL AVENUE,Bronx,47.84,53,0,2026-10-19 00:13:11
3,4616253,CBE E TAYLOR AVENUE - CASTLE HILL AVENUE,Bronx,52.19,48,0,2026-10-19 00:18:10
4,4616253,CBE E TAYLOR AVENUE - CASTLE HILL AVENUE,Bronx,49.70,50,0,2026-10-19 00:23:09
